# Retrieving Chassis Feedback Information

Upon startup, the lower-level systems of the product continuously send various types of feedback information to the upper-level systems. This feedback can be utilized to ascertain the current operational status of the product.

Typically, you would continuously monitor the feedback information from the lower-level systems. However, in this example, we will retrieve a single piece of JSON feedback information from the lower-level systems (you can continuously receive feedback information by commenting out or deleting the break line).

Select the code block below and run it using Ctrl + Enter. The loop will exit and display the feedback information after receiving the first complete JSON message with a "T" value of 1001. The feedback information includes the current wheel speed, IMU data, gimbal angle (if installed), arm angle (if installed), power voltage, and other data.


In [ ]:
from base_ctrl import BaseController
import json
import time

base = BaseController('/dev/ttyTHS1', 115200)

# Implement an infinite loop to continuously monitor serial port data.
while True:
    try:
        # Read a line of data from the serial port, decode it into a 'utf-8' formatted string, and attempt to convert it into a JSON object.
        data_recv_buffer = json.loads(base.rl.readline().decode('utf-8'))
        # Check if the parsed data contains the key 'T'.
        if 'T' in data_recv_buffer:
            # If the value of 'T' is 1001, print the received data and break out of the loop.
            if data_recv_buffer['T'] == 1001:
                print(data_recv_buffer)
                break
    # If an exception occurs while reading or processing the data, ignore the exception and continue to listen for the next line of data.
    except:
        pass

## Non-Blocking Method for Receiving JSON Information via Serial Port

The following code is intended for understanding the underlying principles of reading JSON information from a serial port and should not be executed.

In [ ]:
class ReadLine:
    # Constructor, initializes an instance of the ReadLine class
    # s: The serial port object used to communicate with the serial device.
    def __init__(self, s):
        self.buf = bytearray()  # Initialize a bytearray buffer to store data read from the serial port but not yet processed
        self.s = s              # Store the serial port object for subsequent reading
        self.timeout = 0.1      # Timeout period in seconds; if a complete frame is not received within this time, return None
        self.frame_start = b'{' # Define the start marker of a data frame
        self.frame_end = b"}\r\n" # Define the end marker of a data frame
        self.max_frame_length = 512  # Define the maximum frame length to avoid memory overflow or blocking by malformed data

    def readline(self):
        start_time = time.time()  # Record the time when reading starts, used for timeout checking
        while True:
            # Read as many bytes as possible from the serial port without exceeding max_frame_length
            # Ensure at least 1 byte is read each time
            i = max(1, min(self.max_frame_length, self.s.in_waiting))
            data = self.s.read(i)  # Read i bytes from the serial port
            if data:
                self.buf.extend(data)  # Append the newly read data to the buffer

            # Search for the end marker in the buffer
            end = self.buf.rfind(self.frame_end)

            if end >= 0:  # If the end marker is found
                # Search for the start marker before the end marker
                start = self.buf.rfind(self.frame_start, 0, end)
                if start >= 0 and start < end:
                    # Extract the complete data frame including start and end markers
                    r = self.buf[start:end + len(self.frame_end)]
                    # Remove the processed data from the buffer
                    self.buf = self.buf[end + len(self.frame_end):]
                    return r
                elif start == -1:
                    # Start marker not found, continue reading data
                    continue

            # Handle timeout
            if time.time() - start_time > self.timeout:
                break


This method reads data from a serial port frame by frame, where each frame starts with `{` and ends with `}\r\n`.

* It first checks whether a complete frame already exists in the buffer; if so, it extracts and returns it directly.
* If no complete frame is in the buffer, it reads available bytes from the serial port (up to a maximum of 512 bytes) using `in_waiting`, appending the data to the buffer.
* After each read, it searches for the frame end marker `}\r\n`. If found, it then searches backward for the frame start marker `{`. When both markers exist in the correct order, it extracts this segment as a complete frame, returns it, and removes the processed data from the buffer.
* If no complete frame is found within the configured timeout period (0.1 seconds), the method returns `None`.

### Function Features

* **Strong Frame Parsing Capability**: Accurately identifies complete data frames that start with `{` and end with `}\r\n`, ensuring the parsed data is complete and correctly formatted.
* **Non-blocking Reading**: Uses short timeouts and on-demand reading, avoiding long blocking even when no data is available, making it suitable for real-time systems.
* **Efficient Buffer Management**: Reads data in chunks up to 512 bytes and dynamically appends to a buffer, minimizing memory usage and preventing data loss.
* **Good Fault Tolerance**: Handles leftover incomplete data in the buffer, automatically concatenating and parsing new incoming data to maintain continuity.
* **Strong Real-time Performance**: Immediately checks for and extracts complete frames after each read, suitable for high-frequency serial communication scenarios.
* **Optimized for Structured Data**: Designed to efficiently read, extract, and return structured frame formats like JSON, reducing complexity in subsequent data processing.